In [0]:
%sql
USE flights_gold;

In [0]:
%python
from pyspark.sql.functions import col, lit

airlines_silver = "flights_silver.dim_airlines"
airports_silver = "flights_silver.dim_airports"
flights_silver = "flights_silver.flights"
gold_fact = "flights_gold.fact_flights"

# ładowanie przyrostowe
max_timestamp = None
if spark.catalog.tableExists(gold_fact):
    try:
        # maksymalna data już w tabeli
        row = spark.sql(f"SELECT MAX(scheduled_departure) AS max_date FROM {gold_fact}").collect()[0]
        max_timestamp = row["max_date"]
    except Exception:
        pass

# wczytanie faktów z silver
df_flights = spark.table(flights_silver)
if max_timestamp:
    df_flights_to_process = df_flights.filter(col("scheduled_departure") > max_timestamp)
else:
    df_flights_to_process = df_flights

if df_flights_to_process.count() == 0:
    print("There's no new data to process.")
else:
    # join z tabelami wymiarów
    df_airlines = spark.table(airlines_silver)
    df_airports = spark.table(airports_silver)

    # airlines
    df_joined = df_flights_to_process.alias("f").join(
        df_airlines.alias("al"),
        on=[
            col("f.airline") == col("al.iata_code"), # łączymy po kodzie linii lotniczej
            # lot odbył się w trakcie używania danej nazwy linii (w przedziale start_date - end_date)
            (col("f.scheduled_departure") >= col("al.start_date")) & 
            ((col("f.scheduled_departure") < col("al.end_date")) | (col("al.end_date").isNull()))
        ],
        how="left"
    )

    # airports (origin)
    df_joined = df_joined.join(
        df_airports.alias("ap_org"),
        on=[
            col("f.origin_airport") == col("ap_org.iata_code"),
            (col("f.scheduled_departure") >= col("ap_org.start_date")) & 
            ((col("f.scheduled_departure") < col("ap_org.end_date")) | (col("ap_org.end_date").isNull()))
        ],
        how="left"
    )
    
    # airport (destinantion)
    df_joined = df_joined.join(
        df_airports.alias("ap_dest"),
        on=[
            col("f.destination_airport") == col("ap_dest.iata_code"),
            # tutaj zamiast scheduled departure używamy scheduled_arrival -> bo dolatujemy dopiero na to lotnisko
            (col("f.scheduled_arrival") >= col("ap_dest.start_date")) & 
            ((col("f.scheduled_arrival") < col("ap_dest.end_date")) | (col("ap_dest.end_date").isNull()))
        ],
        how="left"
    )

    # zapis do tabeli faktów
    df_gold_fact = df_joined.select(
        col("f.year"), 
        col("f.month"), 
        col("f.day"), 
        col("f.day_of_week"),
        col("f.flight_number"), 
        col("f.tail_number"),
        col("f.airline").alias("airline_code"),
        col("al.airline_name"), 
        col("f.origin_airport").alias("origin_code"),
        col("ap_org.airport_name").alias("origin_airport_name"),
        col("ap_org.city").alias("origin_city"),
        col("f.destination_airport").alias("dest_code"),
        col("ap_dest.airport_name").alias("dest_airport_name"),
        col("ap_dest.city").alias("dest_city"),
        col("f.scheduled_departure"),
        col("f.departure_time"),
        col("f.wheels_off"),
        col("f.wheels_on"),
        col("f.scheduled_arrival"),
        col("f.arrival_time"),
        col("f.scheduled_time"),
        col("f.elapsed_time"),
        col("f.air_time"),
        col("f.distance"),
        col("f.taxi_out"),
        col("f.taxi_in"),
        col("f.departure_delay"),
        col("f.arrival_delay"),
        col("f.cancelled"),
        col("f.cancellation_reason"),
        col("f.diverted"),
        col("f.air_system_delay"),
        col("f.security_delay"),
        col("f.airline_delay"),
        col("f.late_aircraft_delay"),
        col("f.weather_delay")
    )

    # zapis przyrostowy (APPEND)
    df_gold_fact.write.format("delta").mode("append").saveAsTable(gold_fact)

In [0]:
display(df_gold_fact.limit(5))

year,month,day,day_of_week,flight_number,tail_number,airline_code,airline_name,origin_code,origin_airport_name,origin_city,dest_code,dest_airport_name,dest_city,scheduled_departure,departure_time,wheels_off,wheels_on,scheduled_arrival,arrival_time,scheduled_time,elapsed_time,air_time,distance,taxi_out,taxi_in,departure_delay,arrival_delay,cancelled,cancellation_reason,diverted,air_system_delay,security_delay,airline_delay,late_aircraft_delay,weather_delay
2015,7,3,5,307,N345NB,DL,Delta Air Lines Inc.,BHM,Birmingham-Shuttlesworth International Airport,Birmingham,ATL,Hartsfield-Jackson Atlanta International Airport,Atlanta,2015-07-03T10:35:00.000Z,2015-07-07T00:35:00.000Z,2015-07-03T12:19:00.000Z,2015-07-03T13:52:00.000Z,2015-07-03T12:32:00.000Z,2015-07-07T01:32:00.000Z,57,56,33,134,18,5,86,85,0,null,0,0,0,14,71,0
2015,7,3,5,3581,N518MQ,MQ,American Eagle Airlines Inc.,LGA,LaGuardia Airport (Marine Air Terminal),New York,RDU,Raleigh-Durham International Airport,Raleigh,2015-07-03T10:35:00.000Z,2015-07-06T14:35:00.000Z,2015-07-03T12:09:00.000Z,2015-07-03T13:13:00.000Z,2015-07-03T12:18:00.000Z,2015-07-05T23:18:00.000Z,103,86,64,431,18,4,76,59,0,null,0,0,0,17,42,0
2015,7,3,5,652,N489UA,UA,United Air Lines Inc.,BOS,Gen. Edward Lawrence Logan International Airport,Boston,DEN,Denver International Airport,Denver,2015-07-03T10:35:00.000Z,2015-07-03T08:35:00.000Z,2015-07-03T10:44:00.000Z,2015-07-03T12:34:00.000Z,2015-07-03T13:11:00.000Z,2015-07-02T05:11:00.000Z,276,246,230,1754,11,5,-2,-32,0,null,0,0,0,0,0,0
2015,7,3,5,766,N526VA,VX,Virgin America,DAL,Dallas Love Field,Dallas,LGA,LaGuardia Airport (Marine Air Terminal),New York,2015-07-03T10:35:00.000Z,2015-07-03T03:35:00.000Z,2015-07-03T10:46:00.000Z,2015-07-03T14:45:00.000Z,2015-07-03T14:55:00.000Z,2015-07-03T15:55:00.000Z,200,208,179,1381,18,11,-7,1,0,null,0,0,0,0,0,0
2015,7,3,5,874,N522VA,VX,Virgin America,LAX,Los Angeles International Airport,Los Angeles,DAL,Dallas Love Field,Dallas,2015-07-03T10:35:00.000Z,2015-07-03T07:35:00.000Z,2015-07-03T10:40:00.000Z,2015-07-03T15:28:00.000Z,2015-07-03T15:40:00.000Z,2015-07-03T20:40:00.000Z,185,193,168,1246,8,17,-3,5,0,null,0,0,0,0,0,0
